# 🤖 TelecomX LATAM — Modelo Predictivo de Evasión de Clientes (Churn)

**Challenge 3 — Data Science LATAM | Alura**

---

## 🎯 Objetivo

Construir un modelo de **Machine Learning** capaz de predecir si un cliente de TelecomX va a cancelar su servicio (**Churn**), utilizando variables demográficas, de servicios contratados y de facturación.

## 🗂️ Flujo del Notebook

| Etapa | Descripción |
|-------|-------------|
| **1. Carga de datos** | Extracción desde API + pipeline ETL |
| **2. Exploración** | Análisis visual del Churn |
| **3. Preprocesamiento** | Encoding, escalado y división de datos |
| **4. Modelo** | Entrenamiento de Regresión Logística |
| **5. Evaluación** | Métricas, matriz de confusión e importancia de variables |
| **6. Conclusiones** | Interpretación y recomendaciones |

## 📦 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import requests
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías importadas correctamente')

---
## 📡 2. Carga de Datos

Los datos se extraen directamente desde la API de GitHub y se procesan con el mismo pipeline ETL del proyecto anterior, garantizando consistencia y reproducibilidad sin depender de archivos locales.

In [ ]:
URL = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science-LATAM/main/TelecomX_Data.json"

print(f'📡 Extrayendo datos desde la API...')
response = requests.get(URL)
response.raise_for_status()
data = response.json()

# Normalizar JSON jerárquico a DataFrame tabular
df_raw = pd.json_normalize(data)
print(f'✅ Datos extraídos: {df_raw.shape[0]} registros, {df_raw.shape[1]} columnas')

In [ ]:
# ── Pipeline ETL ──────────────────────────────────────────────
df = df_raw.copy()

# 1. Renombrar columnas
df.columns = df.columns.str.replace('.', '_', regex=False)
df.rename(columns={
    'customerID':                 'cliente_id',
    'Churn':                      'evasion',
    'customer_gender':            'genero',
    'customer_SeniorCitizen':     'adulto_mayor',
    'customer_Partner':           'tiene_pareja',
    'customer_Dependents':        'tiene_dependientes',
    'customer_tenure':            'meses_cliente',
    'phone_PhoneService':         'servicio_telefono',
    'phone_MultipleLines':        'lineas_multiples',
    'internet_InternetService':   'servicio_internet',
    'internet_OnlineSecurity':    'seguridad_online',
    'internet_OnlineBackup':      'backup_online',
    'internet_DeviceProtection':  'proteccion_dispositivo',
    'internet_TechSupport':       'soporte_tecnico',
    'internet_StreamingTV':       'streaming_tv',
    'internet_StreamingMovies':   'streaming_peliculas',
    'account_Contract':           'tipo_contrato',
    'account_PaperlessBilling':   'factura_digital',
    'account_PaymentMethod':      'metodo_pago',
    'account_Charges_Monthly':    'cargo_mensual',
    'account_Charges_Total':      'cargo_total'
}, inplace=True)

# 2. Convertir tipos
df['cargo_total']   = pd.to_numeric(df['cargo_total'],   errors='coerce')
df['cargo_mensual'] = pd.to_numeric(df['cargo_mensual'], errors='coerce')
df['meses_cliente'] = pd.to_numeric(df['meses_cliente'], errors='coerce')

# 3. Limpiar valores inválidos en evasion y nulos
df = df[df['evasion'].isin(['Yes', 'No'])]
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# 4. Eliminar columna ID (no aporta al modelo)
df.drop(columns=['cliente_id'], inplace=True)

print(f'✅ Pipeline ETL completado: {df.shape[0]} registros listos')
df.head()

---
## 🔍 3. Exploración Visual del Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Exploración de la Variable Objetivo — Churn', fontsize=14, fontweight='bold')

# Distribución general
conteo = df['evasion'].value_counts()
colores = ['#2ECC71', '#E74C3C']
axes[0].pie(conteo.values, labels=['No Evadió', 'Evadió'], colors=colores,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Distribución General')

# Por tipo de contrato
tasa_contrato = df.groupby('tipo_contrato')['evasion'].apply(
    lambda x: (x == 'Yes').mean() * 100)
axes[1].bar(tasa_contrato.index, tasa_contrato.values,
            color=['#E74C3C' if v > 20 else '#3498DB' for v in tasa_contrato.values],
            edgecolor='white')
axes[1].set_title('Tasa de Evasión por Contrato')
axes[1].set_ylabel('Tasa (%)')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(tasa_contrato.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# Por tipo de internet
tasa_internet = df.groupby('servicio_internet')['evasion'].apply(
    lambda x: (x == 'Yes').mean() * 100)
axes[2].bar(tasa_internet.index, tasa_internet.values,
            color=['#E74C3C' if v > 20 else '#3498DB' for v in tasa_internet.values],
            edgecolor='white')
axes[2].set_title('Tasa de Evasión por Internet')
axes[2].set_ylabel('Tasa (%)')
axes[2].tick_params(axis='x', rotation=10)
for i, v in enumerate(tasa_internet.values):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## ⚙️ 4. Preprocesamiento para Machine Learning

Antes de entrenar el modelo necesitamos:
1. **Codificar** la variable objetivo (`evasion`) como 0/1
2. **Aplicar One-Hot Encoding** a las variables categóricas
3. **Dividir** en conjuntos de entrenamiento y prueba
4. **Escalar** las variables numéricas

In [ ]:
# Codificar variable objetivo
df['evasion'] = df['evasion'].map({'Yes': 1, 'No': 0})

print('✅ Variable objetivo codificada:')
print(df['evasion'].value_counts())

In [ ]:
# One-Hot Encoding para variables categóricas
df_model = pd.get_dummies(df, drop_first=True)

print(f'✅ Encoding aplicado: {df.shape[1]} → {df_model.shape[1]} columnas')
print(f'   Nuevas columnas generadas: {df_model.shape[1] - df.shape[1]}')
df_model.head(3)

In [ ]:
# Mapa de correlación con la variable objetivo
correlaciones = df_model.corr()['evasion'].drop('evasion').sort_values()

plt.figure(figsize=(10, 7))
colores_corr = ['#E74C3C' if v > 0 else '#2ECC71' for v in correlaciones.values]
plt.barh(correlaciones.index, correlaciones.values, color=colores_corr, edgecolor='white')
plt.axvline(x=0, color='gray', linewidth=0.8)
plt.title('Correlación de Variables con Evasión (Churn)', fontsize=13, fontweight='bold')
plt.xlabel('Correlación')
plt.tight_layout()
plt.show()

In [ ]:
# Separar features (X) y target (y)
X = df_model.drop('evasion', axis=1)
y = df_model['evasion']

# División entrenamiento / prueba (70% / 30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'✅ División completada:')
print(f'   Entrenamiento: {X_train.shape[0]} registros ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'   Prueba:        {X_test.shape[0]} registros ({X_test.shape[0]/len(X)*100:.0f}%)')

In [ ]:
# Escalado de variables numéricas
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('✅ Escalado aplicado con StandardScaler')

---
## 🧠 5. Entrenamiento del Modelo

Se utiliza **Regresión Logística**, un modelo interpretable y efectivo para clasificación binaria como la predicción de Churn.

In [ ]:
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_sc, y_train)

y_pred      = modelo.predict(X_test_sc)
y_pred_prob = modelo.predict_proba(X_test_sc)[:, 1]

print('✅ Modelo entrenado y predicciones generadas')

---
## 📊 6. Evaluación del Modelo

In [ ]:
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_prob)

print('=' * 45)
print('       MÉTRICAS DE EVALUACIÓN DEL MODELO')
print('=' * 45)
print(f'  Accuracy (Exactitud):  {acc:.4f}  ({acc*100:.1f}%)')
print(f'  ROC-AUC Score:         {auc:.4f}')
print('=' * 45)
print()
print('Reporte de Clasificación:')
print(classification_report(y_test, y_pred, target_names=['No Evadió', 'Evadió']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Evaluación del Modelo — Regresión Logística', fontsize=14, fontweight='bold')

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Evadió', 'Evadió'],
            yticklabels=['No Evadió', 'Evadió'],
            linewidths=0.5)
axes[0].set_title('Matriz de Confusión')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Valor Real')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='#3498DB', lw=2,
             label=f'ROC-AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Modelo aleatorio')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#3498DB')
axes[1].set_title('Curva ROC')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 variables más influyentes en el modelo
coef = pd.Series(modelo.coef_[0], index=X.columns)
top15 = coef.abs().nlargest(15).index
coef_top = coef[top15].sort_values()

plt.figure(figsize=(10, 6))
colores_coef = ['#E74C3C' if v > 0 else '#2ECC71' for v in coef_top.values]
plt.barh(coef_top.index, coef_top.values, color=colores_coef, edgecolor='white')
plt.axvline(x=0, color='gray', linewidth=0.8)
plt.title('Top 15 Variables con Mayor Impacto en la Predicción de Churn',
          fontsize=12, fontweight='bold')
plt.xlabel('Coeficiente del Modelo\n(rojo = aumenta riesgo | verde = reduce riesgo)')
plt.tight_layout()
plt.show()

---
## 📄 7. Interpretación y Conclusiones

### Desempeño del Modelo

El modelo de **Regresión Logística** alcanzó una exactitud superior al **80%** y un ROC-AUC por encima de **0.84**, lo que indica una buena capacidad para distinguir entre clientes que evadirán y los que no.

| Métrica | Interpretación |
|---------|---------------|
| **Accuracy > 80%** | El modelo clasifica correctamente a más del 80% de los clientes |
| **ROC-AUC > 0.84** | Alta capacidad de discriminación entre clases |
| **Recall en clase Churn** | El modelo detecta una parte relevante de los clientes que evadirán |

### Variables con Mayor Impacto

Según los coeficientes del modelo, los factores que **aumentan** el riesgo de evasión son:

- 🔴 **Contrato Month-to-Month** — la variable de mayor riesgo
- 🔴 **Servicio de Fibra Óptica** — asociado a mayor insatisfacción
- 🔴 **Cargo mensual alto** — clientes que pagan más tienden a irse más
- 🔴 **Pago por cheque electrónico** — perfil de cliente menos comprometido

Los factores que **reducen** el riesgo de evasión son:

- 🟢 **Mayor antigüedad como cliente** — la lealtad acumulada retiene
- 🟢 **Contratos de 1 o 2 años** — generan mayor compromiso
- 🟢 **Soporte técnico y seguridad online contratados** — aumentan la dependencia del servicio

### Recomendaciones de Negocio

1. **Usar el modelo para identificar clientes en riesgo** y activar campañas de retención proactivas
2. **Ofrecer contratos anuales** con descuento a clientes que están en mes a mes hace más de 6 meses
3. **Mejorar la propuesta de fibra óptica** en precio o calidad de servicio
4. **Incentivar servicios adicionales** (soporte, seguridad) que aumentan la retención
5. **Priorizar el onboarding** en los primeros 12 meses, donde el riesgo de evasión es más alto